# Applicant Consolidation in PATSTAT Explorer

**Audience:** PATSTAT analysts / patent domain experts
**Purpose:** This document explains the methodology behind the "Applicant Search" and
applicant consolidation — what it does, *why* it does it that way, and which SQL queries
actually run against PATSTAT. It is intended as training material.


In [5]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT
patstat = PatstatClient(env='PROD')

def timed_query(query):
    """Execute query and return DataFrame with timing."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    print(f"Query took {time.time() - start:.2f}s ({len(res)} rows)")
    return pd.DataFrame(res)

## 1. The problem: PATSTAT knows no companies, only names

PATSTAT performs **no entity resolution**. A single organization appears in the
`tls206_person` table as many distinct records — one `person_id` per spelling, per
country, per subsidiary, per typo.

Example "Siemens Healthineers" (training example, 21 hits):

In [4]:
df_a = timed_query("""
SELECT p.person_name AS name,
       p.person_ctry_code AS country,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a ON pa.appln_id = a.appln_id
WHERE pa.applt_seq_nr > 0
    AND UPPER(p.person_name) LIKE 'SIEMENS HEALTHINEERS%'
    AND a.appln_filing_year BETWEEN 2014 AND 2024   -- only if year filter is set
GROUP BY p.person_name, p.person_ctry_code
ORDER BY families DESC
LIMIT 200
""")
df_a

Query took 2.87s (21 rows)


,name,country,families
0,Siemens Healthineers AG,DE,1017
1,SIEMENS HEALTHINEERS AG,DE,579
2,Siemens Healthineers International AG,CH,192
3,SIEMENS HEALTHINEERS INTERNATIONAL AG,CH,53
4,SIEMENS HEALTHINEERS INTERNATIONAL AG,US,18
5,"Siemens Healthineers Endovascular Robotics, Inc.",US,15
6,SIEMENS HEALTHINEERS NEDERLAND B.V.,NL,13
7,Siemens Healthineers Nederland B.V.,NL,9
8,SIEMENS HEALTHINEERS AG,,8
9,Siemens Healthineers Ltd.,CN,4


Anyone querying just **one** of these names dramatically underestimates the portfolio.
Anyone naively lumping all hits for a search term together may mix in unrelated companies
or subsidiaries. **Applicant consolidation** is the controlled middle path:
search broadly → group deliberately → evaluate consolidated.

---

## 2. The three tables that carry everything

The entire methodology rests on a three-table join:

```
tls206_person          tls207_pers_appln              tls201_appln
(who)                  (which role, which appln)      (the application)
─────────────          ─────────────────────────      ──────────────
person_id        ───►  person_id
person_name            appln_id              ───►     appln_id
person_ctry_code       applt_seq_nr  (applicant >0)   docdb_family_id
                       invt_seq_nr   (inventor >0)    appln_filing_year
                                                      appln_auth
```

- **`tls206_person`** — the names and country codes. This is where the variants live.
- **`tls207_pers_appln`** — the person↔application link table. The field
  `applt_seq_nr > 0` denotes **applicant role**. `invt_seq_nr > 0` would be the
  inventor role. We filter on `applt_seq_nr > 0` everywhere.
- **`tls201_appln`** — the application itself: family ID, filing year, filing authority.

### Why always `COUNT(DISTINCT docdb_family_id)`?

The same invention is often filed in several countries (DE, US, CN, EP, WO …). All of
these filings share **one** `docdb_family_id` (DOCDB patent family). If we counted
applications, a patent filed broadly internationally would inflate the result.
**The patent family is the correct counting unit for "how many inventions".**
That is why *every* query in the pipeline uses `COUNT(DISTINCT a.docdb_family_id)`.

---

## 3. The workflow in three stages

```
 Stage 1            Stage 1.5                  Stage 2 (optional)    Stage 3
 ────────           ─────────                  ──────────────────    ───────
 Find               Group + REVIEW             Trend preview         Consolidated
 variants      ──►  (suggestion, then     ──►  (sanity check)   ──►  deep analysis
 (LIKE-prefix)       human cleans up)          (1 chart)             (trend/jurisd./CPC)
```

All queries run through the same path:
`Browser → POST /api/query → SvelteKit validation → sidecar.py → PATSTAT BigQuery (PROD)`.

---

### Stage 1 — Variant discovery

You enter a search term (e.g. `Siemens Healthineers`) and optionally a filing-year
range. From this, **one** query is generated that lists all name variants together with
their family count:



Important details for interpretation:

- **Prefix search.** It's a `LIKE 'SEARCHTERM%'` — only names that *start with* the
  search term. "Healthineers Siemens" or "X Siemens Healthineers" would **not** be
  found. Pick a search term that appears at the beginning of the name where possible.
- **Case-insensitive.** Input is normalized to upper case and compared with
  `UPPER(person_name)`. `siemens` also finds `SIEMENS`.
- **Year filter is optional.** It's only appended if the range differs from the default
  (1970–2024). Note: in this stage the year filter affects the *hit list* — a variant
  with 0 families in the time window will not appear.
- **Cap at 200.** `LIMIT 200`, sorted by family count descending. Very broad search
  terms (e.g. just "Siemens") can therefore truncate variants in the long tail.

Result: the hit list on the left in the UI, each row with country and family count.

---

### Stage 1.5 — Group & review (the actual consolidation decision)

There is **no SQL** here — this is the human or heuristic decision about which variants
are the same organization. The stage is **two-step**: first a suggestion, then a binding
review. Only the second part closes it out.

There are three ways to produce the suggestion:

1. **Manually** — check/uncheck boxes, "All" / "None".
2. **Auto-suggest** — a heuristic groups variants with the same "core name".
3. **Display name (parent)** — the name the group will appear under later.
   Default is the variant with the most families ("most filings"); alternatively a
   freely entered "Custom name" (e.g. "Siemens Healthineers Group").

#### How "Auto-suggest" determines the core name

Legal-form and filler words are stripped from each name; the **first** content-bearing
word is kept. Comparison then runs on this single core word.

Removed (among others, list `LEGAL_SUFFIXES`):

> AG, GMBH, GMBH&CO, INC, LTD, LIMITED, CO, CORP, CORPORATION, SA, NV, BV, SE, PLC,
> LLC, KG, OHG, SRL, SPA, AB, AS, OY, PTY, LP, LLP, AKTIENGESELLSCHAFT, GESELLSCHAFT,
> MBH, KABUSHIKI, KAISHA, KK, COMPANY, INDUSTRIES, INDUSTRIAL, ELECTRONICS, ELECTRIC,
> INTERNATIONAL, HOLDINGS, GROUP, GLOBAL, OF, THE, AND, DE, DER, DES, ET, UND

**Property of the method (not a bug):** Only the **first** core word is considered —
`Siemens Healthineers AG` → core `SIEMENS`. Auto-suggest therefore deliberately groups
*broadly*: "Siemens AG" or "Siemens Mobility" are also suggested. This is intentional —
better to suggest too broadly than to miss variants. Precisely for this reason the
grouping is two-step and the following review step is mandatory.

#### Mandatory step: review & clean up the selection

Auto-suggest only provides a suggestion. Before you click "Preview Trend" or "Analyse
Consolidated", you walk through the suggestion — this is part of the method, not an
optional polish:

1. **Check the sector:** variants that clearly belong to a different division
   (e.g. "Siemens Energy", "Siemens Mobility" when searching for "Siemens
   Healthineers") — uncheck them.
2. **Inspect countries / family counts:** classify conspicuous single hits in
   unexpected countries deliberately (genuine foreign subsidiary vs. unrelated company).
3. **Set the display name:** pick the most specific matching variant or a custom name.
4. **Sanity check:** "Preview Trend" — the consolidated family count must be
   **≤ the sum** of the individual family counts (rule of thumb, see section 5).

Only the cleaned-up selection feeds into the deep analysis. Manual control is the gold
standard; auto-suggest is the starting point, not the result.

---

### Stage 2 — Trend preview (optional sanity check)

"Preview Trend" validates the selection **before** you go into the deep analysis. One
query that returns the consolidated family count per year:


In [8]:
df_a = timed_query("""
SELECT a.appln_filing_year AS year,
COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE p.person_name IN ('Siemens Healthineers AG','SIEMENS HEALTHINEERS AG')
  AND pa.applt_seq_nr > 0
  AND a.appln_filing_year BETWEEN 1990 AND 2024
GROUP BY a.appln_filing_year
ORDER BY year
""")
df_a

Query took 2.64s (14 rows)


,year,families
0,2008,1
1,2012,7
2,2013,9
3,2014,20
4,2015,9
5,2016,34
6,2017,38
7,2018,53
8,2019,86
9,2020,68


The consolidation itself is visible here: the many selected variants are reduced to
**one `IN` list**. `COUNT(DISTINCT docdb_family_id)` across all variants ensures that a
family filed under two name variants of the same corporate group is counted **only
once** — this is the real value-add of consolidation compared to "just adding up the
families of the variants".

---

### Stage 3 — Consolidated deep analysis

The selected names are handed over to the analysis view. A WHERE filter is built from
them:

- **Exactly one name:** `p.person_name = 'NAME'`
- **Multiple names:** `p.person_name IN ('NAME1','NAME2', …)`

Three queries then run in parallel:




In [15]:
### **3a — Filing trend (families per year)**

df_a = timed_query("""
SELECT a.appln_filing_year AS year,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE p.person_name IN ('Siemens Healthineers AG','SIEMENS HEALTHINEERS AG')
  AND pa.applt_seq_nr > 0
  AND a.appln_filing_year BETWEEN 1990 AND 2024
GROUP BY a.appln_filing_year
ORDER BY year
""")
df_a


Query took 0.35s (14 rows)


,year,families
0,2008,1
1,2012,7
2,2013,9
3,2014,20
4,2015,9
5,2016,34
6,2017,38
7,2018,53
8,2019,86
9,2020,68


In [16]:
### **3b — Top filing authorities / jurisdictions**

df_b = timed_query("""
SELECT a.appln_auth AS authority,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE p.person_name IN ('Siemens Healthineers AG','SIEMENS HEALTHINEERS AG')
  AND pa.applt_seq_nr > 0
GROUP BY a.appln_auth
ORDER BY families DESC
LIMIT 15
""")
df_b

Query took 0.43s (8 rows)


,authority,families
0,US,808
1,EP,567
2,DE,474
3,WO,15
4,JP,7
5,ES,2
6,KR,1
7,CN,1


In [17]:
### **3c — Top technology fields (CPC 4-digit)**

df_c = timed_query("""
SELECT SUBSTR(c.cpc_class_symbol, 1, 4) AS cpc,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
JOIN tls224_appln_cpc c ON a.appln_id = c.appln_id
WHERE p.person_name IN ('Siemens Healthineers AG','SIEMENS HEALTHINEERS AG')
  AND pa.applt_seq_nr > 0
GROUP BY cpc
ORDER BY families DESC
LIMIT 15
""")
df_c

Query took 0.40s (15 rows)


,cpc,families
0,A61B,789
1,G06T,486
2,G01R,456
3,G16H,366
4,G06N,188
5,G06V,160
6,G06F,147
7,H05G,42
8,G01T,41
9,G01N,41


All three use the same consolidated name filter — the consolidation decision from
Stage 1.5 is therefore carried consistently through every analytical dimension.

---

## 4. End-to-end example: "Siemens Healthineers"

1. Search term `Siemens Healthineers`, years 2014–2024 → **21 hits** (Stage 1).
2. "All" selects everything initially; via auto-suggest/manual narrowed down to the
   genuine Healthineers entities → **16 names grouped**.
3. Display name = `Siemens Healthineers AG` (most filings, 1,017 families).
4. "Preview Trend" shows: **1,920 families total** across the 16 variants —
   *consolidated*, i.e. without double-counting across families.
5. "Analyse Consolidated" → trend, jurisdiction and CPC charts for the group.

> Note: 1,920 is **less than** the naive sum of all variant families, because families
> carried under multiple variants count only once. This is precisely the evidence that
> consolidation is working.

---

## 5. Pitfalls & quality control

| Risk | Symptom | Countermeasure |
|---|---|---|
| **Over-consolidation** | Auto-suggest only matches the 1st core word → unrelated divisions get included | Manually uncheck variants; review countries / family counts |
| **Under-consolidation** | Subsidiary with a different name stem is missing (e.g. "Endovascular Robotics") | Run several search terms, add hits manually |
| **Prefix boundary** | Variant doesn't start with the search term | Shorten the search term / target the start of the name |
| **`LIMIT 200`** | Very broad search term, long-tail variants missing | Make the search term more specific |
| **Year filter in Stage 1** | Variant with filings only outside the window is missing from the list | Keep the filter wide initially; narrow only in the analysis |
| **Subsidiary = its own company?** | Methodological question, not a data question | Decide consciously in the training context and document |

**Sanity-check rule of thumb:** The consolidated family count (Stage 2/3) is always
**≤ the sum of individual family counts** from the hit list. If it *equals* the sum,
there were no cross-variant family overlaps (rare for genuine corporate groups) —
that's a hint to revisit the selection.

---

## 6. Technical appendix (quick reference)

- **API path:** `POST /api/query` (SvelteKit) → `sidecar.py /api/query` →
  `PatstatClient(env="PROD").sql_query(sql, use_legacy_sql=False)` (Google BigQuery
  Standard SQL).
- **Safety guard:** `SELECT` only, max. 10,000 characters, no multi-statements.
- **Source code:** Stage 1 & 2 in `src/routes/search/+page.svelte`; Stage 3 in
  `src/routes/applicant/+page.svelte`; name-filter logic in `src/lib/context.ts`
  (`nameWhereClause`).